In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content =" This is a simple page needed for demonstrating RAG pipeline",
    metadata={
        "source":"This is from the sample document",
        "author":"Arun Sridhar",
        "pages": 1,
        "Date Created":"2024-01-01"
    }

)


print(doc.page_content)
print(doc.metadata)


 This is a simple page needed for demonstrating RAG pipeline
{'source': 'This is from the sample document', 'author': 'Arun Sridhar', 'pages': 1, 'Date Created': '2024-01-01'}


In [3]:
from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/doc.txt")
documents = loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/doc.txt'}, page_content="Cricket is a bat-and-ball sport for two teams of 11 players, where one team bats to score runs, and the other bowls and fields to restrict runs and get batters out (take wickets). The goal is to score more runs than the opposing team to win. Key elements include a central pitch with two wickets, a bowler delivering a ball, and a batter using a cricket bat to strike it, aiming to hit it into the field to score runs or hit it across the boundary for four or six runs. The game has diverse formats, from the multi-day Test matches to the exciting shorter T20 games, each offering a unique experience for players and spectators alike. \nOrigins and History\nCricket originated in England, possibly in the 16th century, and became a national sport in the 18th century. \nThe British Empire expanded the game's reach globally, introducing it to countries like India in the early 1700s. \nOver time, the sport evolved from an el

In [5]:
####Directory Loading
from langchain.document_loaders import DirectoryLoader
loader = DirectoryLoader("../data/text_files", 
                            glob="**/*.txt",
                            loader_cls=TextLoader,
                            show_progress=True,
                            loader_kwargs={"encoding": "utf8"}
                        )
documents = loader.load()
print(documents)    

100%|██████████| 2/2 [00:00<00:00, 976.10it/s]

[Document(metadata={'source': '../data/text_files/doc.txt'}, page_content="Cricket is a bat-and-ball sport for two teams of 11 players, where one team bats to score runs, and the other bowls and fields to restrict runs and get batters out (take wickets). The goal is to score more runs than the opposing team to win. Key elements include a central pitch with two wickets, a bowler delivering a ball, and a batter using a cricket bat to strike it, aiming to hit it into the field to score runs or hit it across the boundary for four or six runs. The game has diverse formats, from the multi-day Test matches to the exciting shorter T20 games, each offering a unique experience for players and spectators alike. \nOrigins and History\nCricket originated in England, possibly in the 16th century, and became a national sport in the 18th century. \nThe British Empire expanded the game's reach globally, introducing it to countries like India in the early 1700s. \nOver time, the sport evolved from an el

In [16]:
####Directory Loading
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
dir_loader = DirectoryLoader("../data/pdfs", 
                            glob="**/*.pdf",
                            loader_cls=PyPDFLoader,
                            show_progress=True
                        )
pdf_documents = dir_loader.load()
print(pdf_documents)

100%|██████████| 2/2 [00:02<00:00,  1.10s/it]

[Document(metadata={'source': '../data/pdfs/Redis-Microservices-for-Dummies.pdf', 'page': 0}, page_content=''), Document(metadata={'source': '../data/pdfs/Redis-Microservices-for-Dummies.pdf', 'page': 1}, page_content='These materials are © 2020 John Wiley & Sons, Inc. Any dissemination, distribution, or unauthorized use is strictly prohibited.'), Document(metadata={'source': '../data/pdfs/Redis-Microservices-for-Dummies.pdf', 'page': 2}, page_content='These materials are © 2020 John Wiley & Sons, Inc. Any dissemination, distribution, or unauthorized use is strictly prohibited.\nRedis \nMicroservices'), Document(metadata={'source': '../data/pdfs/Redis-Microservices-for-Dummies.pdf', 'page': 3}, page_content='These materials are © 2020 John Wiley & Sons, Inc. Any dissemination, distribution, or unauthorized use is strictly prohibited.\nRedis \nMicroservices\nLimited Edition\nby Kyle Davis with Loris Cro'), Document(metadata={'source': '../data/pdfs/Redis-Microservices-for-Dummies.pdf', 

In [ ]:
#Embedding and Vector Store
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Optional,Tuple
from sklearn.metrics.pairwise import cosine_similarity




In [ ]:
class EmbeddingManager:
    """
    Handles document embedding generation using SentenceTransformers.
    Args: model_name : str : Name of the pre-trained SentenceTransformer model to use.
    """
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = None #SentenceTransformer(model_name)
        self.model_name = model_name
        self._load_model()
        
    def _load_model(self):
        """_summary_: Loads the SentenceTransformer model
        """
        try:
            print(f"Loading model {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        
    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generates embeddings for a list of texts.
        Args:
            texts (List[str]): List of texts to embed.
        Returns:
            np.ndarray: Array of embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generating embeddings with shape: {embeddings.shape} texts...")

        return embeddings
    
    def get_embeddings_dimension(self) -> int:
        """
        Returns the dimension of the embeddings.
        Returns:
            int: Dimension of the embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        return self.model.get_sentence_embedding_dimension()
    
    
##intialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager
    
    
    

In [28]:
from sentence_transformers import SentenceTransformer
    # Load a pre-trained model (e.g., 'all-mpnet-base-v2')
model = SentenceTransformer('all-mpnet-base-v2')
    # Sentences to encode
sentences = ['This is an example sentence', 'Each sentence is converted']
    # Encode sentences to get their embeddings
sentence_embeddings = model.encode(sentences)
print(sentence_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 0.0225026  -0.07829176 -0.02303072 ... -0.00827928  0.02652685
  -0.00201898]
 [ 0.04170233  0.00109742 -0.01553417 ... -0.02181629 -0.06359359
  -0.00875287]]


In [31]:
import os
class VectorStore:
    """_summary_: Manages a ChromaDB vector store for storing and retrieving document embeddings.
    Args:
        collection_name (str): Name of the ChromaDB collection.
        persist_directory (str): Directory to persist the ChromaDB database.
        embedding_dimension (int): Dimension of the embeddings.
    """
    
    def __init__(self, 
                 collection_name: str = 'pdf_documents', 
                 persist_directory: str = '../data/vector_store', 
                 embedding_dimension: int = 384):
        self.client = None
        self.collection = None
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.embedding_dimension = embedding_dimension
        self._initialize_store()
        
    def _initialize_store(self):
        """_summary_: Initializes the ChromaDB client and collection.
        """
        try:
            print("Initializing ChromaDB client...")
            os.makedirs(self.persist_directory, exist_ok=True)
            
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            
            # self.client = chromadb.Client(Settings(
            #     chroma_db_impl="duckdb+parquet",
            #     persist_directory=self.persist_directory
            # ))
            print("ChromaDB client initialized.")
            
            # Create or get the collection
            if self.collection_name in [col.name for col in self.client.list_collections()]:
                self.collection = self.client.get_collection(name=self.collection_name)
                print(f"Collection '{self.collection_name}' loaded.")
            else:
                self.collection = self.client.create_collection(name=self.collection_name, 
                                                                metadata={"embedding_dimension": self.embedding_dimension})
                print(f"Collection '{self.collection_name}' created.")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e
            
    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Adds documents and their embeddings to the ChromaDB collection.
        Args:
            documents (List[Document]): List of Document objects.
            embeddings (np.ndarray): Array of embeddings corresponding to the documents.
        """
        if not self.collection:
            raise ValueError("ChromaDB collection not initialized.")
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        ids = [str(uuid.uuid4()) for _ in range(len(documents))]
        metadatas = [doc.metadata for doc in documents]
        texts = [doc.page_content for doc in documents]
        
        print(f"Adding {len(documents)} documents to the collection...")
        self.collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            metadatas=metadatas,
            documents=texts
        )
        print("Documents added successfully.")
        
vector_store = VectorStore()
vector_store
vector_store.add_documents(pdf_documents, 
                           embedding_manager.generate_embeddings(
                               [doc.page_content for doc in pdf_documents]
                           )
                       )
vector_store

Initializing ChromaDB client...
Error initializing ChromaDB: You are using a deprecated configuration of Chroma.

If you do not have data you wish to migrate, you only need to change how you construct
your Chroma client. Please see the "New Clients" section of https://docs.trychroma.com/migration.
________________________________________________________________________________________________

If you do have data you wish to migrate, we have a migration tool you can use in order to
migrate your data to the new Chroma architecture.
Please `pip install chroma-migrate` and run `chroma-migrate` to migrate your data and then
change how you construct your Chroma client.

See https://docs.trychroma.com/migration for more information or join our discord at https://discord.gg/8g5FESbj for help!


ValueError: [91mYou are using a deprecated configuration of Chroma.

[94mIf you do not have data you wish to migrate, you only need to change how you construct
your Chroma client. Please see the "New Clients" section of https://docs.trychroma.com/migration.
________________________________________________________________________________________________

If you do have data you wish to migrate, we have a migration tool you can use in order to
migrate your data to the new Chroma architecture.
Please `pip install chroma-migrate` and run `chroma-migrate` to migrate your data and then
change how you construct your Chroma client.

See https://docs.trychroma.com/migration for more information or join our discord at https://discord.gg/8g5FESbj for help![0m